# COMPASS RAG Evaluation Notebook

Evaluates the COMPASS OUD Research Assistant's retrieval-augmented generation pipeline across four dimensions:

| Metric | What it measures |
|--------|------------------|
| **Context Relevance** | Are the retrieved chunks actually useful for the question? |
| **Faithfulness** | Does the answer stay grounded in the retrieved context (no hallucination)? |
| **Answer Relevance** | Does the answer actually address what was asked? |
| **Source Coverage** | Does the answer cite the expected topic area? |

Evaluation is done two ways:
1. **RAGAS** — automated framework using LLM-as-judge  
2. **Custom Claude judge** — fine-grained domain scoring with chain-of-thought

---
**Requirements:** `pip install -r requirements.txt`  
**COMPASS API** must be running at `COMPASS_URL` (default: `http://localhost:8000` or set the env var).  
**ANTHROPIC_API_KEY** must be set in your environment or `.env` file.

## 1. Setup

In [ ]:
import json, os, time, sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
import anthropic

# Load .env from backend folder if present
REPO_ROOT = Path("../").resolve()
load_dotenv(REPO_ROOT / "compass-app" / "backend" / ".env")

# Config
COMPASS_URL  = os.getenv("COMPASS_URL", "https://compass.axiomsystemslab.com")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY", "")
EVAL_MODEL   = "claude-sonnet-4-6"   # model used as the judge
RESULTS_DIR  = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

if not ANTHROPIC_KEY:
    print("⚠️  ANTHROPIC_API_KEY not set — Claude judge scoring will be skipped.")

print(f"COMPASS endpoint : {COMPASS_URL}")
print(f"Judge model      : {EVAL_MODEL}")
print(f"Results folder   : {RESULTS_DIR.resolve()}")

In [ ]:
# Quick health check
try:
    r = requests.get(f"{COMPASS_URL}/health", timeout=10)
    health = r.json()
    print("✅ COMPASS is reachable")
    print(f"   Index ready  : {health.get('index_ready')}")
    print(f"   Chain loaded : {health.get('chain_loaded')}")
    print(f"   Active model : {health.get('llm_backend')}")
except Exception as e:
    print(f"❌ Could not reach COMPASS: {e}")
    print("   Start the server, or set COMPASS_URL to the correct address.")

## 2. Load Test Dataset

In [ ]:
with open("test_set.json") as f:
    test_set = json.load(f)

df_test = pd.DataFrame(test_set)
print(f"Loaded {len(df_test)} test questions")
print(f"Topics: {df_test['topic'].nunique()}")
print(f"Difficulty breakdown:")
print(df_test['difficulty'].value_counts().to_string())
df_test[["id", "topic", "difficulty", "question"]].style.set_properties(**{"text-align": "left"})

## 3. Query COMPASS API

In [ ]:
def query_compass(question: str, timeout: int = 120) -> dict:
    """Send a question to COMPASS and return {answer, sources, backend, latency_s}."""
    t0 = time.time()
    try:
        r = requests.post(
            f"{COMPASS_URL}/chat",
            json={"question": question},
            timeout=timeout,
        )
        r.raise_for_status()
        data = r.json()
        return {
            "answer":    data.get("answer", ""),
            "sources":   data.get("sources", []),
            "backend":   data.get("backend", "unknown"),
            "latency_s": round(time.time() - t0, 2),
            "error":     None,
        }
    except Exception as e:
        return {"answer": "", "sources": [], "backend": "error",
                "latency_s": round(time.time() - t0, 2), "error": str(e)}

In [ ]:
# Run all questions — this may take a few minutes depending on the LLM backend
results = []

for item in tqdm(test_set, desc="Querying COMPASS"):
    resp = query_compass(item["question"])
    results.append({
        "id":             item["id"],
        "topic":          item["topic"],
        "difficulty":     item["difficulty"],
        "question":       item["question"],
        "expected_themes":item["expected_themes"],
        "answer":         resp["answer"],
        "sources":        resp["sources"],
        "backend":        resp["backend"],
        "latency_s":      resp["latency_s"],
        "error":          resp["error"],
    })
    time.sleep(0.5)   # be polite to the server

df = pd.DataFrame(results)
df["answered"] = df["error"].isna() & (df["answer"].str.len() > 20)
df["n_sources"] = df["sources"].apply(len)

print(f"\n✅ {df['answered'].sum()}/{len(df)} questions answered successfully")
print(f"Average latency : {df['latency_s'].mean():.1f}s")
print(f"Average sources : {df['n_sources'].mean():.1f}")

# Save raw results
raw_path = RESULTS_DIR / f"raw_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(raw_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Raw results saved to {raw_path}")

## 4. Source Coverage (Keyword-Based)

Check whether the answer mentions the expected keywords/themes for each question.
Fast, deterministic, no API call needed.

In [ ]:
def keyword_coverage(answer: str, themes: list[str]) -> dict:
    """Fraction of expected themes found in the answer (case-insensitive)."""
    ans_lower = answer.lower()
    found = [t for t in themes if t.lower() in ans_lower]
    return {
        "found":    found,
        "missing":  [t for t in themes if t.lower() not in ans_lower],
        "score":    round(len(found) / len(themes), 3) if themes else 0.0,
    }

df["kw"] = df.apply(lambda r: keyword_coverage(r["answer"], r["expected_themes"]), axis=1)
df["kw_score"] = df["kw"].apply(lambda x: x["score"])

print("Keyword coverage scores per question:")
display(df[["id", "topic", "difficulty", "kw_score", "n_sources"]].sort_values("kw_score"))

## 5. Claude-as-Judge Evaluation

Uses Claude to score each Q&A pair on three axes (0–5 each):

| Dimension | Definition |
|-----------|------------|
| **Faithfulness** | Are all claims in the answer grounded in the cited sources / retrieved context? |
| **Answer Relevance** | Does the answer directly address what was asked, without significant off-topic content? |
| **Completeness** | Does the answer cover the key aspects expected for this type of clinical/policy question? |

In [ ]:
JUDGE_PROMPT = """\
You are evaluating an AI assistant called COMPASS that answers questions about opioid use disorder (OUD) treatment, policy, and research.

Score the following Q&A on three dimensions from 0 to 5:

- **Faithfulness (0-5)**: Are all factual claims in the answer grounded in evidence? Does the answer avoid hallucinating clinical details not supported by OUD research? (5 = fully grounded, no hallucinations; 0 = fabricated claims)
- **Answer Relevance (0-5)**: Does the answer directly and completely address the question asked? (5 = directly addresses all parts; 0 = off-topic or evasive)
- **Completeness (0-5)**: Does the answer cover the key clinical/policy concepts expected for this question type? (5 = comprehensive; 0 = missing critical information)

Cited sources: {sources}

Question: {question}

Answer: {answer}

Respond ONLY with valid JSON in this exact format:
{{"faithfulness": <int>, "answer_relevance": <int>, "completeness": <int>, "reasoning": "<one sentence>"}}
"""

def judge_with_claude(row: dict) -> dict:
    """Score one Q&A row using Claude. Returns dict with scores or None on error."""
    if not ANTHROPIC_KEY:
        return None
    if not row.get("answered", True):
        return {"faithfulness": 0, "answer_relevance": 0, "completeness": 0, "reasoning": "No answer returned."}

    source_titles = ", ".join(
        s.get("file", "unknown") for s in (row.get("sources") or [])
    ) or "none cited"

    prompt = JUDGE_PROMPT.format(
        sources=source_titles,
        question=row["question"],
        answer=row["answer"][:2000],   # truncate very long answers
    )

    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    try:
        msg = client.messages.create(
            model=EVAL_MODEL,
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}],
        )
        text = msg.content[0].text.strip()
        # Strip markdown code fences if present
        if text.startswith("```"):
            text = "\n".join(text.split("\n")[1:-1])
        return json.loads(text)
    except Exception as e:
        print(f"Judge error on {row['id']}: {e}")
        return None

In [ ]:
# Run Claude judge — skipped automatically if no API key
scores = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Claude judge"):
    score = judge_with_claude(row.to_dict())
    scores.append(score)
    time.sleep(0.3)

df["judge_raw"] = scores
df["faithfulness"]     = df["judge_raw"].apply(lambda x: x["faithfulness"]     if x else None)
df["answer_relevance"] = df["judge_raw"].apply(lambda x: x["answer_relevance"] if x else None)
df["completeness"]     = df["judge_raw"].apply(lambda x: x["completeness"]     if x else None)
df["judge_reasoning"]  = df["judge_raw"].apply(lambda x: x.get("reasoning", "") if x else "")

# Composite score (average of 3 dimensions, normalised to 0-1)
judge_cols = ["faithfulness", "answer_relevance", "completeness"]
df["composite_score"] = df[judge_cols].mean(axis=1) / 5.0

scored = df["composite_score"].notna().sum()
print(f"\nJudge scoring complete: {scored}/{len(df)} questions scored")
if scored > 0:
    print(f"\nMean scores (0–5):")
    print(df[judge_cols].mean().round(2).to_string())
    print(f"\nComposite (0–1): {df['composite_score'].mean():.3f}")

## 6. RAGAS Evaluation (Optional)

RAGAS provides a standardised evaluation framework for RAG pipelines.  
Requires `ragas`, `langchain-anthropic`, and an Anthropic API key.

In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from ragas.llms import LangchainLLMWrapper
    from langchain_anthropic import ChatAnthropic
    from datasets import Dataset
    RAGAS_AVAILABLE = True
    print("✅ RAGAS available")
except ImportError as e:
    RAGAS_AVAILABLE = False
    print(f"⚠️  RAGAS not available: {e}")
    print("   Run: pip install ragas langchain-anthropic datasets")

In [ ]:
if RAGAS_AVAILABLE and ANTHROPIC_KEY:
    # Build RAGAS dataset from results
    ragas_data = []
    for _, row in df[df["answered"]].iterrows():
        contexts = [
            f"[{s.get('file','')} p.{s.get('page','')}] {s.get('excerpt','')}"
            for s in (row["sources"] or [])
        ]
        if contexts:   # RAGAS needs at least one context
            ragas_data.append({
                "question":  row["question"],
                "answer":    row["answer"],
                "contexts":  contexts,
            })

    ragas_dataset = Dataset.from_list(ragas_data)

    # Configure Claude as the judge LLM
    llm = LangchainLLMWrapper(ChatAnthropic(model=EVAL_MODEL, api_key=ANTHROPIC_KEY))

    for m in [faithfulness, answer_relevancy, context_precision]:
        m.llm = llm

    print(f"Running RAGAS on {len(ragas_dataset)} Q&A pairs…")
    ragas_result = evaluate(
        ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    ragas_df = ragas_result.to_pandas()
    print("\n=== RAGAS Summary ===")
    print(ragas_df[["faithfulness", "answer_relevancy", "context_precision"]].describe().round(3))

    ragas_path = RESULTS_DIR / f"ragas_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    ragas_df.to_csv(ragas_path, index=False)
    print(f"RAGAS results saved to {ragas_path}")
else:
    print("RAGAS evaluation skipped (install ragas + set ANTHROPIC_API_KEY to enable).")

## 7. Results & Visualisation

In [ ]:
# Full results table
display_cols = ["id", "topic", "difficulty", "latency_s", "n_sources", "kw_score",
                "faithfulness", "answer_relevance", "completeness", "composite_score"]
available_cols = [c for c in display_cols if c in df.columns]

styled = (
    df[available_cols]
    .style
    .background_gradient(subset=[c for c in ["kw_score","faithfulness","answer_relevance","completeness","composite_score"] if c in available_cols],
                         cmap="RdYlGn")
    .format({
        "latency_s":       "{:.1f}s",
        "kw_score":        "{:.2f}",
        "composite_score": "{:.2f}",
    }, na_rep="—")
)
display(styled)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("COMPASS RAG Evaluation Results", fontsize=14, fontweight="bold", y=1.01)
sns.set_style("darkgrid")
pal = sns.color_palette("husl", 4)

# 1. Keyword coverage by topic
ax = axes[0, 0]
topic_kw = df.groupby("topic")["kw_score"].mean().sort_values()
bars = ax.barh(topic_kw.index, topic_kw.values, color=pal[0], alpha=0.85)
ax.set_xlim(0, 1)
ax.set_xlabel("Mean keyword coverage score")
ax.set_title("Keyword Coverage by Topic")
ax.axvline(0.7, color="orange", linestyle="--", linewidth=1, label="0.7 target")
ax.legend(fontsize=8)

# 2. Latency distribution
ax = axes[0, 1]
ax.hist(df["latency_s"].dropna(), bins=10, color=pal[1], alpha=0.85, edgecolor="white")
ax.axvline(df["latency_s"].mean(), color="red", linestyle="--", label=f"mean={df['latency_s'].mean():.1f}s")
ax.set_xlabel("Latency (seconds)")
ax.set_ylabel("Count")
ax.set_title("Response Latency Distribution")
ax.legend(fontsize=8)

# 3. Judge scores heatmap (if available)
ax = axes[1, 0]
judge_cols_present = [c for c in ["faithfulness","answer_relevance","completeness"] if c in df.columns and df[c].notna().any()]
if judge_cols_present:
    pivot = df.set_index("id")[judge_cols_present].dropna()
    sns.heatmap(pivot, ax=ax, cmap="RdYlGn", vmin=0, vmax=5,
                annot=True, fmt=".0f", linewidths=0.5, cbar_kws={"label": "Score (0–5)"})
    ax.set_title("Claude Judge Scores per Question")
    ax.set_xlabel("")
else:
    ax.text(0.5, 0.5, "Claude judge scores\nnot available",
            ha="center", va="center", transform=ax.transAxes, fontsize=11, color="gray")
    ax.set_title("Claude Judge Scores (N/A)")

# 4. Sources cited per question
ax = axes[1, 1]
ax.bar(df["id"], df["n_sources"], color=pal[3], alpha=0.85)
ax.set_xlabel("Question ID")
ax.set_ylabel("Sources cited")
ax.set_title("Sources Cited per Answer")
ax.tick_params(axis="x", rotation=45, labelsize=7)
ax.axhline(df["n_sources"].mean(), color="orange", linestyle="--",
           label=f"mean={df['n_sources'].mean():.1f}")
ax.legend(fontsize=8)

plt.tight_layout()
plot_path = RESULTS_DIR / f"eval_charts_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Charts saved to {plot_path}")
plt.show()

## 8. Summary Report

In [ ]:
print("=" * 60)
print("COMPASS RAG EVALUATION SUMMARY")
print(f"Timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Endpoint   : {COMPASS_URL}")
print(f"Backend    : {df['backend'].mode()[0] if not df.empty else 'unknown'}")
print(f"Questions  : {len(df)}")
print("=" * 60)
print(f"\n📥 RETRIEVAL")
print(f"  Avg sources cited      : {df['n_sources'].mean():.2f}")
print(f"  Questions w/ 0 sources : {(df['n_sources']==0).sum()}")

print(f"\n📝 KEYWORD COVERAGE (proxy for topic recall)")
print(f"  Mean score             : {df['kw_score'].mean():.3f}")
print(f"  Questions ≥ 0.7        : {(df['kw_score'] >= 0.7).sum()}/{len(df)}")

judge_present = df["composite_score"].notna().any() if "composite_score" in df.columns else False
if judge_present:
    print(f"\n🤖 CLAUDE JUDGE (0–5)")
    for col in ["faithfulness", "answer_relevance", "completeness"]:
        if col in df.columns:
            print(f"  {col:<22} : {df[col].mean():.2f}")
    print(f"  Composite (0–1)        : {df['composite_score'].mean():.3f}")

print(f"\n⏱️  PERFORMANCE")
print(f"  Mean latency           : {df['latency_s'].mean():.1f}s")
print(f"  Max latency            : {df['latency_s'].max():.1f}s")

print(f"\n⚠️  WEAKEST QUESTIONS (lowest keyword coverage):")
worst = df.nsmallest(5, "kw_score")[["id", "topic", "kw_score", "question"]]
for _, r in worst.iterrows():
    print(f"  [{r['id']}] kw={r['kw_score']:.2f}  {r['question'][:70]}…")

# Save full results CSV
csv_path = RESULTS_DIR / f"eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
save_cols = [c for c in ["id","topic","difficulty","question","answer","n_sources",
              "latency_s","kw_score","faithfulness","answer_relevance",
              "completeness","composite_score","judge_reasoning","backend"] if c in df.columns]
df[save_cols].to_csv(csv_path, index=False)
print(f"\n✅ Full results saved to {csv_path}")

## 9. Per-Question Deep Dive

Inspect any single question in detail — answer, sources, and judge reasoning.

In [ ]:
# Change QUESTION_ID to any id from the test set
QUESTION_ID = "T1-03"   # low-dose buprenorphine / Bernese method

row = df[df["id"] == QUESTION_ID].iloc[0]

print(f"Question [{row['id']}]  |  {row['topic']}  |  {row['difficulty']}")
print("-" * 70)
print(f"Q: {row['question']}")
print(f"\nLatency: {row['latency_s']}s  |  Sources: {row['n_sources']}  |  KW score: {row['kw_score']:.2f}")

if "composite_score" in row and pd.notna(row.get("composite_score")):
    print(f"Judge: faithfulness={row['faithfulness']}  relevance={row['answer_relevance']}  completeness={row['completeness']}")
    print(f"Reasoning: {row['judge_reasoning']}")

print(f"\n--- ANSWER ---\n{row['answer']}")

print(f"\n--- SOURCES ({row['n_sources']}) ---")
for s in row["sources"]:
    print(f"  [{s.get('topic','')}] {s.get('file','')} p.{s.get('page','')}")
    print(f"    {s.get('excerpt','')[:120]}…")